In [73]:
!pip install pandas tqdm

In [74]:
months = {
 "січня": "01",
 "лютого": "02",
 "березня": "03",
 "квітня": "04",
 "травня": "05",
 "червня": "06",
 "липня": "07",
 "серпня": "08",
 "вересня": "09",
 "жовтня": "10",
 "листопада": "11",
 "грудня": "12"
}

In [75]:
currencies = {
 "грн": "UAH",
 "₴": "UAH",
 "доларів": "USD",
 "долари": "USD",
 "$": "USD",
 "євро": "EUR",
 "€": "EUR"
}

In [76]:
cities = [
    "сша",
    "італія",
    "україна",
    "франція",
    "німеччина",
    "угорщина",
    "росія",
    "рф",
    "японія",
    "австралія",
    "австрія",
    "білорусь",
    "румунія",
    "британія",
    "великобританія",
    "індія",
    "польща",
    "іспанія",
    "португалія",
    "чехія",
    "словаччина",
    "болгарія",
    "греція",
    "туреччина",
    "литва",
    "латвія",
    "естонія",
    "молдова",
    "швеція",
    "швеції",
    "італії",
    "азейбарджан",
    "україни",
    "франції",
    "німеччини",
    "угорщини",
    "росії",
    "рф",
    "японії",
    "австралії",
    "австрії",
    "білорусі",
    "румунії",
    "британії",
    "великобританії",
    "індії",
    "польщі",
    "іспанії",
    "португалії",
    "чехії",
    "словаччини",
    "болгарії",
    "греції",
    "туреччини",
    "литви",
    "латвії",
    "естонії",
    "молдови"
]

In [77]:
# IE RULES
import re
import json
from pathlib import Path

city_pattern = r"\b(" + "|".join(cities) + r")\b"

date_numeric = re.compile(r"\b\d{1,2}[./]\d{1,2}[./]\d{4}\b")
date_text = re.compile(
    r"\b(\d{1,2})\s+(січня|лютого|березня|квітня|травня|червня|липня|серпня|вересня|жовтня|листопада|грудня)\s+(\d{4})"
)

amount_pattern = re.compile(
    r"(\d+)\s?(млн|тис)?\s?(грн|₴|доларів|євро|\$|€)"
)


def normalize_date_numeric(text):
    day, month, year = re.split("[./]", text)
    return f"{year}-{int(month):02d}-{int(day):02d}"


def normalize_date_text(day, month, year):
    month_num = months.get(month)
    if month_num:
        return f"{year}-{month_num}-{int(day):02d}"
    return None


def extract_dates(text):

    results = []

    for m in date_numeric.finditer(text):

        span = m.group()
        norm = normalize_date_numeric(span)

        results.append({
            "field_type": "DATE",
            "value": norm,
            "span_text": span,
            "start_char": m.start(),
            "end_char": m.end(),
            "method": "regex_date_numeric"
        })

    for m in date_text.finditer(text):

        day, month, year = m.groups()

        norm = normalize_date_text(day, month, year)

        results.append({
            "field_type": "DATE",
            "value": norm,
            "span_text": m.group(),
            "start_char": m.start(),
            "end_char": m.end(),
            "method": "regex_date_text"
        })

    return results


def extract_amounts(text):

    results = []

    for m in amount_pattern.finditer(text):

        number = int(m.group(1))
        multiplier = m.group(2)
        currency = currencies.get(m.group(3))

        if multiplier == "млн":
            number *= 1_000_000

        if multiplier == "тис":
            number *= 1_000

        results.append({
            "field_type": "AMOUNT",
            "value": number,
            "currency": currency,
            "span_text": m.group(),
            "start_char": m.start(),
            "end_char": m.end(),
            "method": "regex_amount"
        })

    return results


def extract_locations(text):

    results = []

    for m in re.finditer(city_pattern, text):

        results.append({
            "field_type": "LOCATION",
            "value": m.group(),
            "span_text": m.group(),
            "start_char": m.start(),
            "end_char": m.end(),
            "method": "city_dictionary"
        })

    return results


def extract_all(text):

    results = []

    results.extend(extract_dates(text))
    results.extend(extract_amounts(text))
    results.extend(extract_locations(text))

    return results

In [78]:
def evaluate_precision(predictions, gold):

    correct = 0
    predicted = 0

    for p, g in zip(predictions, gold):

        pred_entities = p["entities"]
        gold_entities = g["annotations"]

        predicted += len(pred_entities)

        for pe in pred_entities:

            for ge in gold_entities:

                if pe["span_text"] == ge["span_text"]:
                    correct += 1

    precision = correct / predicted if predicted else 0

    return precision

In [79]:
def evaluate_on_file(file_path):
    with open(file_path, encoding="utf8") as f:
        gold = json.load(f)

    predictions = []
    for entry in gold:
        text = entry["text"]
        extracted = extract_all(text)
        predictions.append({"entities": extracted})

    return evaluate_precision(predictions, gold)

In [80]:
import pandas as pd
from tqdm import tqdm
import numpy as np
import re
from sklearn.metrics import precision_score, recall_score, f1_score

df = pd.read_csv("/content/processed_v2.csv")

print("Dataset:", df.shape)
df.head()

print(df.columns)

df = df.rename(columns={"processed_text": "text"})

print(df.columns)
df.head()

Dataset: (12956, 2)
Index(['processed_text', 'label'], dtype='object')
Index(['text', 'label'], dtype='object')


,text,label
0,десятки бригад! операція почалась – новий наст...,Fake
1,дуже важкий грип зараз мандрує україною! у діт...,Fake
2,"виcтyп гeнceкa нато nідірвaв мережу: “цiнa, як...",Fake
3,"залишилися лічені дні, почнеться справжня “м'я...",Fake
4,"кремль втретє змінив тактику щодо україни, теп...",Fake


In [81]:
df = df.sample(1000, random_state=42)

texts = df["text"].tolist()

In [82]:
all_results = []

for text in tqdm(texts):

    entities = extract_all(text)

    all_results.append({
        "text": text,
        "entities": entities
    })

100%|██████████| 1000/1000 [00:00<00:00, 3656.87it/s]


In [83]:
print(all_results[0])

{'text': 'у 2011 році нато, під керівництвом сша, порушив лівію та ... у 2011 році нато, під керівництвом сша, порушив лівію і, отже, допомагав ісіс отримати вплив там.зараз нова кампанія бомбардувань дуже ймовірна.насправді це допоможе лише isis, аналогічно, коли сша допомагають їм у сирії.', 'entities': [{'field_type': 'LOCATION', 'value': 'сша', 'span_text': 'сша', 'start_char': 35, 'end_char': 38, 'method': 'city_dictionary'}, {'field_type': 'LOCATION', 'value': 'сша', 'span_text': 'сша', 'start_char': 96, 'end_char': 99, 'method': 'city_dictionary'}, {'field_type': 'LOCATION', 'value': 'сша', 'span_text': 'сша', 'start_char': 255, 'end_char': 258, 'method': 'city_dictionary'}]}


In [84]:
date_count = 0
amount_count = 0
location_count = 0

for r in all_results:

    for e in r["entities"]:

        if e["field_type"] == "DATE":
            date_count += 1

        if e["field_type"] == "AMOUNT":
            amount_count += 1

        if e["field_type"] == "LOCATION":
            location_count += 1

print("DATE:", date_count)
print("AMOUNT:", amount_count)
print("LOCATION:", location_count)

DATE: 87
AMOUNT: 52
LOCATION: 2029


In [85]:
import random
import json

sample_size = 20

filtered_df = df[df["text"].str.len() <= 250]

sample_df = filtered_df.sample(sample_size, random_state=42)

gold_candidates = []

for i, row in sample_df.iterrows():

    gold_candidates.append({
        "text_id": int(i),
        "text": row["text"],
        "annotations": []
    })

with open("gold_candidates.json", "w", encoding="utf8") as f:

    json.dump(gold_candidates, f, ensure_ascii=False, indent=2)

print("Gold subset file created")

Gold subset file created


In [86]:
precision = evaluate_on_file("lab4_gold_ie.json")
print("Precision:", precision)

Precision: 1.0


In [87]:
audit = []

audit.append("# Lab 4 Audit Report\n")

audit.append("## Dataset")
audit.append(f"Texts processed: {len(texts)}")

audit.append("\n## Extracted Entities")

audit.append(f"DATE: {date_count}")
audit.append(f"AMOUNT: {amount_count}")
audit.append(f"LOCATION: {location_count}")

audit.append("\n## Precision")

audit.append(f"Precision: {precision:.3f}")

with open("audit_summary_lab4.md","w") as f:

    f.write("\n".join(audit))

print("Audit file created")

Audit file created
